# The SNN's "Senses" and "Thoughts" (the "Bat Senses")

We are building a supervised classifier. We will show the network two spike trains (the "problem") and tell it the answer (the "label"). After seeing enough examples, it will learn to find the answer on its own.

1. The Inputs (What the SNN "Sees")

The SNN will have two separate inputs that it processes simultaneously over time. Think of it like a musician listening to two audio tracks at once.

Input 1: sent_spikes

What it is: A 1D tensor (a list) of 0s and 1s.

Dimensions: [time_steps]

Meaning: This is the "reference" track. It's the exact, clean spike train our "SNN Transmitter" (from Notebook 4) created. It tells the SNN, "Here is the pattern I sent out, starting at t=0."

Input 2: recovered_spikes

What it is: Another 1D tensor of 0s and 1s.

Dimensions: [time_steps]

Meaning: This is the "echo" track. It's the pattern we heard back after it bounced off the wall and was re-encoded into spikes. It is a delayed, noisy, and slightly different version of sent_spikes.

The SNN's fundamental task is to learn the time-lag between these two tracks.

2. The Output (What the SNN "Guesses")

We will treat this as a classification problem, as it's much simpler for an SNN to learn than precise regression.

The "Bins": We will divide our world (e.g., 1m to 10m) into 10 distinct "bins" or "classes":

Class 0: 1-2 meters

Class 1: 2-3 meters

...

Class 9: 9-10 meters

The SNN Output: spike_counts_per_class

What it is: A 1D tensor with 10 values.

Dimensions: [num_classes] (e.g., 10)

Meaning: The SNN will have 10 "output neurons," one for each bin. The SNN's "guess" is the neuron that fires the most spikes over the simulation time.

Example: If we feed it a 3.5m echo, and the SNN output is [1, 5, 22, 9, 3, 0, 0, 0, 0, 0], the 3rd neuron (index 2) fired the most (22 spikes). The SNN's guess is "Class 2" (i.e., 3-4 meters).

3. The Label (What we "Teach")

This is the "ground truth" we use to correct the SNN's mistakes.

Label: target_label

What it is: A single integer.

Meaning: The correct bin index.

Example: If we generate a sample where the wall is at 3.5m, the correct label is 2. We use this integer to calculate the loss and teach the SNN, "Your guess (Class 2) was correct!" or "Your guess (Class 4) was wrong, the answer was 2."

### Cell 1: Setup & Data Generation Function

This is the most complex cell. It combines our previous work into one optimized function generate_batch. This function will create a batch of training examples (e.g., 32 at a time), each with a different, random wall distance.

In [1]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import time

# --- 1. Simulation Parameters ---
fs = 10000        # Sampling Rate (Hz)
T = 0.2           # Duration of simulation (seconds) - Longer for more context
t_vec = torch.linspace(0, T, int(fs*T))
time_steps = len(t_vec)

# --- 2. SNN Transmitter (Bio-Source) Params ---
tx_beta = 0.95
tx_thresh = 1.0
tx_current = 0.05

# --- 3. Physics & Channel Params ---
c = 343  # Speed of sound (m/s)
min_dist = 1.0  # Min distance to detect
max_dist = 9.0  # Max distance to detect
num_classes = int(max_dist - min_dist) # 8 classes: [1-2], [2-3]...[8-9]

# --- 4. Radar Waveform (Pre-compute filters) ---
chirp_duration = 0.005
bw = 2000
f_start = 500
f_carrier = 4000
t_chirp_np = np.linspace(0, chirp_duration, int(fs*chirp_duration))
chirp_template_np = signal.chirp(t_chirp_np, f0=f_start, f1=f_start+bw, t1=chirp_duration, method='linear')
carrier_np = np.cos(2 * np.pi * f_carrier * t_vec.numpy())
sos_filter = signal.butter(10, f_carrier, 'low', fs=fs, output='sos')
matched_filter = chirp_template_np[::-1]

# --- 5. SNN Receiver (Encoder) Params ---
enc_beta = 0.8
enc_thresh = 0.7
lif_encoder = snn.Leaky(beta=enc_beta, threshold=enc_thresh)

# --- 6. The Data Generation Function ---

def generate_batch(batch_size):
    """Generates a batch of training data."""
    
    # Initialize output tensors
    batch_sent_spikes = torch.zeros(time_steps, batch_size)
    batch_recv_spikes = torch.zeros(time_steps, batch_size)
    batch_labels = torch.zeros(batch_size, dtype=torch.long)

    # Initialize SNN states
    tx_lif = snn.Leaky(beta=tx_beta, threshold=tx_thresh)
    tx_mem = tx_lif.init_leaky()
    
    # --- Generate a unique transmitter spike train FOR THE WHOLE BATCH
    # (This is more realistic; the SNN must learn the delay for *this specific* pulse)
    sent_spikes = torch.zeros(time_steps)
    for t in range(time_steps):
        spk, tx_mem = tx_lif(torch.tensor(tx_current), tx_mem)
        sent_spikes[t] = spk
    
    # --- Create the transmitted RF signal ---
    baseband_np = signal.convolve(sent_spikes.numpy(), chirp_template_np, mode='same')
    tx_signal_np = baseband_np * carrier_np

    # --- Generate each sample in the batch ---
    for i in range(batch_size):
        # 1. Create Label
        dist = np.random.uniform(min_dist, max_dist)
        label = int(dist - min_dist) # e.g., 3.5m -> class 2
        batch_labels[i] = torch.tensor(label, dtype=torch.long)
        
        # 2. Simulate Physics
        tof = (2 * dist) / c
        delay_samples = int(tof * fs)
        
        rx_signal = np.roll(tx_signal_np, delay_samples)
        rx_signal = rx_signal * (1 / (dist + 1)) # Attenuation
        rx_signal += np.random.normal(0, 0.2, len(rx_signal)) # Noise
        
        # 3. Receiver Chain
        demod_raw = rx_signal * carrier_np
        rx_baseband = signal.sosfilt(sos_filter, demod_raw)
        recovered_analog = signal.convolve(rx_baseband, matched_filter, mode='same')
        recovered_current = torch.tensor(recovered_analog).float()
        recovered_current = (recovered_current - recovered_current.min()) / (recovered_current.max() - recovered_current.min()) # Normalize

        # 4. SNN Receiver (Encode echo to spikes)
        enc_mem = lif_encoder.init_leaky()
        recv_spikes_i = torch.zeros(time_steps)
        for t in range(time_steps):
            spk, enc_mem = lif_encoder(recovered_current[t], enc_mem)
            recv_spikes_i[t] = spk
            
        # 5. Store in batch
        batch_sent_spikes[:, i] = sent_spikes
        batch_recv_spikes[:, i] = recv_spikes_i
        
    return batch_sent_spikes, batch_recv_spikes, batch_labels

# Test it:
print(f"SNN Classifier Setup:")
print(f"Time steps: {time_steps} ({T*1000} ms)")
print(f"Num classes: {num_classes} (bins from {min_dist}m to {max_dist}m)")
s, r, l = generate_batch(1) # Generate one sample to test
print(f"Generated sample for dist: {l.item()+min_dist}-{l.item()+min_dist+1}m")
print(f"Sent spikes shape: {s.shape}")
print(f"Recv spikes shape: {r.shape}")

SNN Classifier Setup:
Time steps: 2000 (200.0 ms)
Num classes: 8 (bins from 1.0m to 9.0m)
Generated sample for dist: 1.0-2.0m
Sent spikes shape: torch.Size([2000, 1])
Recv spikes shape: torch.Size([2000, 1])


### Cell 2: Define the SNN Classifier Model

Here is the SNN itself. It's a torch.nn.Module that takes the two spike trains. It has a hidden LIF layer and an output LIF layer. The forward pass loops through time, just like a real brain.

In [2]:
# --- Define the SNN Architecture ---
# Use a fast-sigmoid surrogate gradient
spike_grad = surrogate.fast_sigmoid(slope=25)

class RangeSNN(nn.Module):
    def __init__(self, hidden_size, num_classes):
        super().__init__()
        
        # Layer 1: Input (2 spike inputs) to Hidden
        self.lin1 = nn.Linear(2, hidden_size)
        self.lif1 = snn.Leaky(beta=0.9, threshold=1.0, spike_grad=spike_grad)
        
        # Layer 2: Hidden to Output
        self.lin2 = nn.Linear(hidden_size, num_classes)
        self.lif2 = snn.Leaky(beta=0.9, threshold=1.0, spike_grad=spike_grad)

    def forward(self, sent_spikes, recv_spikes):
        
        # Get batch size from input
        batch_size = sent_spikes.shape[1]
        
        # Initialize hidden states
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        
        # Store output spikes
        spk2_rec = []
        
        # Time-step loop
        for t in range(time_steps):
            # Stack inputs: [batch_size, 2]
            in_t = torch.stack([sent_spikes[t], recv_spikes[t]], dim=1)
            
            # Hidden layer
            cur1 = self.lin1(in_t)
            spk1, mem1 = self.lif1(cur1, mem1)
            
            # Output layer
            cur2 = self.lin2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            spk2_rec.append(spk2)
            
        # Stack all output spikes: [time_steps, batch_size, num_classes]
        all_spikes = torch.stack(spk2_rec)
        
        # Sum spikes over time to get "votes"
        spike_counts = torch.sum(all_spikes, dim=0) # [batch_size, num_classes]
        return spike_counts

# --- Hyperparameters ---
hidden_size = 32
batch_size = 32

# Create the model
net = RangeSNN(hidden_size, num_classes)
print(f"SNN Model created with {hidden_size} hidden neurons.")

SNN Model created with 32 hidden neurons.


### Cell 3: The Training Loop

This is the main training cell. We generate batches of data "on the fly" and use the snntorch surrogate gradient loss function (ce_rate_loss) to train the network.

In [3]:
# --- Training Setup ---
num_epochs = 20
num_iters_per_epoch = 15 # How many batches to generate per epoch
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

# We use Cross-Entropy on *Firing Rate* (spike counts)
loss_fn = snn.functional.ce_rate_loss() 

loss_hist = []
start_time = time.time()

print(f"--- Starting Training ---")
print(f"Epochs: {num_epochs} | Batches/Epoch: {num_iters_per_epoch} | Batch Size: {batch_size}")

for epoch in range(num_epochs):
    epoch_loss = 0
    for i in range(num_iters_per_epoch):
        # 1. Generate a new batch of data
        s_spk, r_spk, labels = generate_batch(batch_size)
        
        # 2. Run the SNN
        net.train() # Set model to training mode
        optimizer.zero_grad()
        spike_counts = net(s_spk, r_spk) # [batch_size, num_classes]
        
        # 3. Calculate loss
        loss = loss_fn(spike_counts, labels)
        
        # 4. Backpropagate (Surrogate Gradients)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / num_iters_per_epoch
    loss_hist.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs} - Avg Loss: {avg_loss:.4f}")

end_time = time.time()
print(f"--- Training Complete ---")
print(f"Total time: {end_time - start_time:.2f}s")

# Plot Loss
plt.figure(figsize=(8, 4))
plt.plot(loss_hist)
plt.title("SNN Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Average Cross-Entropy Loss")
plt.show()

AttributeError: module 'snntorch' has no attribute 'functional'

### Cell 4: Test on Unseen Data

Finally, we test our newly trained SNN. We generate a new, unseen batch of 200 samples and run them through the network. We set model.eval() and use torch.no_grad() to turn off training. We then calculate the final classification accuracy.

In [ ]:
# --- Test Setup ---
num_test_samples = 200
correct = 0
total = 0

print(f"--- Testing on {num_test_samples} unseen samples ---")
net.eval() # Set model to evaluation mode
with torch.no_grad(): # Turn off gradients
    
    # Generate one large test batch
    s_test, r_test, labels_test = generate_batch(num_test_samples)
    
    # Run the SNN
    spike_counts_test = net(s_test, r_test)
    
    # Get the SNN's guess (index of the max spike count)
    _, predictions = spike_counts_test.max(dim=1)
    
    # Compare to ground truth
    total = labels_test.size(0)
    correct = (predictions == labels_test).sum().item()

accuracy = 100 * correct / total
print(f"\n--- Results ---")
print(f"Correct: {correct} / {total}")
print(f"Accuracy on Unseen Data: {accuracy:.2f}%")

# --- Plot a Confusion Matrix ---
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(labels_test.numpy(), predictions.numpy())
class_names = [f"{i+min_dist}-{i+min_dist+1}m" for i in range(num_classes)]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.title('Confusion Matrix (Test Set)')
plt.show()